[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch11-hw-acceleration/02-do-roofline-tren-cpu.ipynb)

# Đo roofline trên CPU của bạn, rồi hợp nhất kernel để bớt lưu lượng

Sổ tay 01 dùng các con số phần cứng của sách. Sổ tay này hỏi một câu khác: trên chính chiếc máy
đang chạy sổ tay, hình dạng mà mô hình roofline dự đoán có hiện ra không? Một CPU không phải A100,
nên các con số ở đây không so được với sách và mỗi máy một khác. Thứ so được là **hình dạng**: một
dốc khi mật độ tính toán thấp, một mái khi mật độ cao, và một chỗ gãy ở giữa.

**Bạn sẽ làm:**
1. đếm mật độ tính toán của hai nguyên thuỷ trong một tầng dày đặc: phép nhân ma trận và phép
   theo từng phần tử;
2. đo thông lượng phép nhân ma trận và băng thông của một phép theo từng phần tử trên máy này;
3. dựng roofline của máy từ hai số đo, rồi đặt lên đó một tầng dày đặc ở chín kích thước batch;
4. đếm lưu lượng của chuỗi ReLU, chuẩn hóa theo batch và affine như bảng 19, 20 của sách, rồi để
   một trình biên dịch hợp nhất nó thành một kernel và đo xem bớt được bao nhiêu.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Hai nguyên thuỷ, hai mật độ

Chương mở đầu phần nguyên thuỷ tính toán bằng một tầng dày đặc 256 đầu vào, 512 đầu ra. Phần nhân
ma trận của nó là phép nhân cộng dồn (MAC) trên mọi cặp đầu vào và trọng số. Phần kích hoạt ReLU
là phép toán vectơ: mỗi phần tử đầu ra một phép so sánh rồi chọn. Ở đây ta dùng FP32, 4 byte mỗi
giá trị, vì đó là kiểu mặc định của PyTorch trên CPU.

In [ ]:
FP32 = 4
vao = sach(256, nguon="256 · 512 · 131,072")
ra = sach(512, nguon="256 · 512 · 131,072")
theo_sach("MAC một lời gọi, tầng 256 → 512", vao * ra, sach=131_072, nguon="256 · 512 · 131,072")
B = sach(32, nguon="32 · 4.2M")
theo_sach("MAC cho batch 32", B * vao * ra, sach=4.2e6, nguon="32 · 4.2M")


def mat_do_nhan(B, M, N, byte=FP32):
    """Phép nhân (B × M) · (M × N): 2BMN FLOP trên số byte đọc đầu vào, đọc trọng số, ghi đầu ra."""
    return 2 * B * M * N / (byte * (B * M + M * N + B * N))


mat_do_relu = 1 / (2 * FP32)    # một phép so sánh, đọc 4 byte và ghi 4 byte
print(f"phần nhân ma trận, batch 32: {mat_do_nhan(32, vao, ra):.2f} FLOP/byte")
print(f"phần ReLU trên {ra} đầu ra: {mat_do_relu} FLOP/byte, bất kể batch")
print(f"chênh nhau {mat_do_nhan(32, vao, ra) / mat_do_relu:.0f} lần trong cùng một tầng")

Hai nguyên thuỷ của cùng một tầng cách nhau hơn hai bậc độ lớn về mật độ. Với phép nhân hai ma
trận vuông $n \times n$, mật độ là $2n^3 / (3n^2 \cdot 4) = n/6$ ở FP32, nên tăng theo $n$. Với
phép theo từng phần tử, nó là một hằng số. Ô dưới kiểm tra điều đó bằng phân số chính xác, không
làm tròn.

In [ ]:
from fractions import Fraction

for n in (16, 128, 1024, 4096):
    assert Fraction(2 * n**3, FP32 * 3 * n * n) == Fraction(n, 6)
    assert Fraction(n, 2 * FP32 * n) == Fraction(1, 8)   # n phép trên n phần tử đọc và ghi
print("nhân ma trận n × n: I = n/6; phép theo từng phần tử: I = 1/8, ở mọi n")

## 2. Đo hai trần của máy này

Một roofline cần hai số: trần tính toán và băng thông. Trên CPU, ta lấy trần tính toán từ phép nhân
ma trận lớn nhất (thư viện của PyTorch chia khối nó rất kỹ), và băng thông từ một phép theo từng
phần tử trên một mảng lớn hơn hẳn cache (mỗi phần tử đọc một lần, ghi một lần).

Mỗi phép được gọi lặp lại đủ lâu, và ta giữ lần nhanh nhất trong vài lượt, để bớt nhiễu từ các
chương trình khác đang chạy cùng máy. Ô thiết lập ở đầu đặt số luồng; trên Colab mặc định là 2.

**Dự đoán:** khi $n$ nhỏ, chẳng hạn 32, phép nhân ma trận đạt bao nhiêu phần trăm thông lượng của
phép nhân lớn nhất? Và vì sao?

In [ ]:
def thoi_gian(f, toi_thieu=0.02, luot=3):
    """Thời gian ngắn nhất cho một lần gọi f (giây): lặp cho tới khi một lượt dài ít nhất `toi_thieu`."""
    f()                                   # khởi động: cấp phát, nạp thư viện
    lap = 1
    while True:
        t0 = time.perf_counter()
        for _ in range(lap):
            f()
        dt = time.perf_counter() - t0
        if dt >= toi_thieu:
            break
        lap *= 2
    tot_nhat = dt / lap
    for _ in range(luot - 1):
        t0 = time.perf_counter()
        for _ in range(lap):
            f()
        tot_nhat = min(tot_nhat, (time.perf_counter() - t0) / lap)
    return tot_nhat


dat_hat_giong()
cac_n = [32, 64, 128, 256, 512, 1024, 2048]
gflops_nhan = []
for n in cac_n:
    A, Bm = torch.randn(n, n), torch.randn(n, n)
    t = thoi_gian(lambda: A @ Bm)
    gflops_nhan.append(2 * n**3 / t / 1e9)

cac_phan_tu = [2**k for k in range(12, 26, 2)] + [2**25]
gbs = []
for so in cac_phan_tu:
    x, y = torch.randn(so), torch.empty(so)
    t = thoi_gian(lambda: torch.mul(x, 2.0, out=y))
    gbs.append(2 * FP32 * so / t / 1e9)

R_may = max(gflops_nhan)        # GFLOP/s
BW_may = gbs[-1]                # GB/s, ở mảng lớn nhất
do_tren_may("trần tính toán (phép nhân lớn nhất)", round(R_may, 1), "GFLOP/s")
do_tren_may("băng thông (mảng lớn nhất)", round(BW_may, 1), "GB/s")
do_tren_may("điểm gãy của máy này", round(R_may / BW_may, 2), "FLOP/byte")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
ax1.plot(cac_n, gflops_nhan, "o-", color="#2a78d6", ms=6)
ax1.set_xscale("log", base=2)
ax1.set_xticks(cac_n, [str(n) for n in cac_n])
ax1.set_ylim(0, R_may * 1.15)
ax1.set_xlabel("cỡ ma trận vuông n")
ax1.set_ylabel("GFLOP/s (đo trên máy này)")
ax1.set_title("Phép nhân ma trận n × n")

ax2.plot([4 * s for s in cac_phan_tu], gbs, "o-", color="#eb6834", ms=6)
ax2.set_xscale("log")
ax2.set_ylim(0, max(gbs) * 1.15)
nhan_truc = [(16e3, "16 KB"), (1e6, "1 MB"), (16e6, "16 MB"), (128e6, "128 MB")]
ax2.set_xticks([v for v, _ in nhan_truc], [s for _, s in nhan_truc])
ax2.set_xlabel("cỡ mảng đầu vào")
ax2.set_ylabel("GB/s đọc và ghi (đo trên máy này)")
ax2.set_title("Phép nhân từng phần tử với 2")
plt.tight_layout()
plt.show()

Cả hai đường đều thấp ở bên trái. Với ma trận nhỏ, mỗi lời gọi tốn một khoản phụ trội cố định
(fixed overhead): gọi từ Python, chọn kernel, chia việc, và khoản ấy không nhỏ đi theo kích thước.
Phép nhân 32 × 32 chỉ có 65,536 FLOP, nên phụ trội chiếm phần lớn thời gian. Đây là số hạng độ trễ
cố định của định luật sắt (iron law), và cũng là một lý do để hợp nhất kernel. Khi kích thước lớn
dần, cả hai đường tiến tới một mức trần: đó là hai trần của roofline trên máy này.

Đường băng thông có thể nhô lên ở giữa rồi hạ xuống một chút. Một mảng vừa với cache được đọc từ
cache, nhanh hơn bộ nhớ chính, và đó là phân cấp bộ nhớ hiện ra trong một phép đo. Vì vậy ta lấy
băng thông ở mảng lớn nhất, nơi dữ liệu chắc chắn phải đi từ bộ nhớ chính.

## 3. Roofline của máy này, và một tầng dày đặc ở chín kích thước batch

Giờ ta lặp lại napkin math 1.8 của sách trên CPU, nhưng **đo** chứ không tính: một tầng dày đặc
$4096 \to 4096$ ở FP32, trọng số chiếm khoảng 67 MB, lớn hơn cache của phần lớn CPU, nên mỗi lần
gọi phải đọc trọng số từ bộ nhớ chính. Mật độ của nó theo batch $B$ là phương trình 6 của sách, với
4 byte mỗi giá trị thay vì 2.

**Dự đoán:** ở batch 1, tầng này là một phép nhân vectơ với ma trận. Nó sẽ nằm trên dốc hay trên
mái của roofline máy này? Còn ở batch 256?

In [ ]:
M = N = 4096
W = torch.randn(M, N)
cac_batch = [1, 2, 4, 8, 16, 32, 64, 128, 256]
I_batch, gflops_batch = [], []
for b in cac_batch:
    X = torch.randn(b, M)
    t = thoi_gian(lambda: X @ W, toi_thieu=0.05)
    I_batch.append(mat_do_nhan(b, M, N))
    gflops_batch.append(2 * b * M * N / t / 1e9)
    do_tren_may(f"batch {b:>3}, I = {I_batch[-1]:6.2f} FLOP/byte", round(gflops_batch[-1], 1), "GFLOP/s")

# Hai điểm tham chiếu: ReLU (I = 1/8) và phép nhân vuông 1024 (I = 1024/6).
so = 2**24
x, y = torch.randn(so), torch.empty(so)
gflops_relu = so / thoi_gian(lambda: torch.clamp(x, min=0.0, out=y)) / 1e9
gflops_1024 = gflops_nhan[cac_n.index(1024)]
assert mat_do_nhan(1, M, N) < 0.5 < mat_do_nhan(2, M, N)   # batch 1 dưới 0.5 FLOP/byte ở FP32

In [ ]:
I_truc = np.logspace(-2, 3, 300)
tran_may = np.minimum(R_may, BW_may * I_truc)
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.loglog(I_truc, tran_may, color="0.35", lw=2, label="roofline từ hai số đo ở phần 2")
ax.loglog(I_batch, gflops_batch, "o-", color="#2a78d6", ms=7, mec="white", mew=1.2,
          label="tầng dày đặc 4096 → 4096, batch 1 tới 256")
ax.loglog([mat_do_relu, 1024 / 6], [gflops_relu, gflops_1024], "s", color="#eb6834", ms=7,
          mec="white", mew=1.2, label="ReLU và nhân ma trận 1024 × 1024")
for b, (dx, dy), ha in ((1, (-4, -15), "center"), (16, (8, -14), "left"), (256, (-12, 10), "right")):
    k = cac_batch.index(b)
    ax.annotate(f"batch {b}", (I_batch[k], gflops_batch[k]), xytext=(dx, dy), textcoords="offset points",
                fontsize=8, ha=ha)
ax.annotate("ReLU", (mat_do_relu, gflops_relu), xytext=(8, -4), textcoords="offset points", fontsize=8)
ax.axvline(R_may / BW_may, color="0.8", lw=1, zorder=0)
ax.set_xlabel("mật độ tính toán I (FLOP/byte, FP32)")
ax.set_ylabel("GFLOP/s (đo trên máy này)")
ax.set_title("Roofline của máy đang chạy sổ tay")
ax.set_ylim(min(gflops_relu, min(gflops_batch)) / 4, R_may * 2.5)
ax.legend(loc="lower right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Hãy đọc hình theo hai phía của vạch dọc, là điểm gãy đo được. Một điểm bên trái vạch chỉ có thể
nhanh tới đường dốc: mỗi byte đọc từ bộ nhớ chính mang theo quá ít phép tính, và đó là vùng nghẽn ở
bộ nhớ. ReLU nằm ở đó trên mọi máy. Một điểm bên phải vạch bị chặn bởi mái, và thêm batch không còn
làm mỗi FLOP rẻ hơn.

Các điểm của tầng dày đặc nằm **dưới** đường xám chứ không trên nó. Roofline là một cận trên, và
khoảng cách tới nó là phần hiệu năng mà kernel chưa lấy được. Với batch nhỏ, mỗi khối trọng số nạp
vào chỉ được dùng cho vài hàng đầu vào, nên kernel nhân ma trận khó giữ các đơn vị tính luôn bận,
kể cả khi băng thông đã thôi là giới hạn. Vì thế một điểm có thể đã qua điểm gãy mà vẫn còn cách
mái khá xa, và chỉ chạm mái khi batch đủ lớn. Roofline nói trần ở đâu; nó không cho biết kernel đạt
được bao nhiêu phần của trần ấy. Ô dưới in phần ấy cho từng batch.

Điểm gãy đo được là của máy này, ở số luồng mà ô thiết lập đã chọn. Sách gọi tỉ số ấy là cân bằng
phần cứng và coi nó là thuộc tính của từng thiết bị: cùng một kernel có thể nghẽn ở bộ nhớ trên
máy này và nghẽn ở tính toán trên máy khác.

In [ ]:
diem_gay_may = R_may / BW_may
for b, I, g in zip(cac_batch, I_batch, gflops_batch):
    phia = "trái" if I < diem_gay_may else "phải"
    do_tren_may(f"batch {b:>3} ({phia} điểm gãy): phần của roofline đạt được",
                round(100 * g / min(R_may, BW_may * I)), "%")

## 4. Hợp nhất kernel: đếm lưu lượng, rồi đo

Chuỗi suy luận của sách: ReLU, rồi chuẩn hóa theo batch (batch normalization) với thống kê đã đóng
băng, rồi phép co giãn affine $Y = \alpha X'' + \beta$. Thực thi trực tiếp chạy ba kernel, và mỗi
kernel ghi ra một tensor trung gian để kernel sau đọc lại. Hợp nhất kernel (kernel fusion) gộp
chúng thành một, để giá trị trung gian đi qua thanh ghi thay vì qua bộ nhớ.

Bảng 19 và 20 của sách đếm điều này cho một tensor 1024 × 1024 ở FP32, với MB là $10^6$ byte.

In [ ]:
MB = 1e6
hang = cot = sach(1024, nguon="1024 × 1024 · 4.2 MB · 16.8 MB")
mot_tensor = hang * cot * FP32
theo_sach("một tensor (MB)", mot_tensor / MB, sach=4.2, nguon="1024 × 1024 · 4.2 MB")
theo_sach("X, X', X'' và Y cùng tồn tại (MB)", 4 * mot_tensor / MB, sach=16.8, nguon="4.2 MB · 16.8 MB")
truc_tiep = (3 + 3) * mot_tensor      # ba lần đọc, ba lần ghi
hop_nhat = (1 + 1) * mot_tensor       # đọc đầu vào một lần, ghi đầu ra một lần
theo_sach("lưu lượng thực thi trực tiếp (MB)", truc_tiep / MB, sach=25.2, nguon="25.2 MB · 8.4 MB · 3 lần")
theo_sach("lưu lượng khi hợp nhất (MB)", hop_nhat / MB, sach=8.4, nguon="25.2 MB · 8.4 MB · 3 lần")
theo_sach("mức giảm", truc_tiep / hop_nhat, sach=3, nguon="25.2 MB · 8.4 MB · 3 lần")

Mức giảm 3 lần là của mô hình lý tưởng hoá: sách nói rõ lưu lượng thực tế còn tuỳ hành vi cache
và việc trình biên dịch dùng lại bộ đệm. Ba phép này đều có mật độ dưới 1 FLOP/byte, nên chúng nằm
sâu trên dốc của mọi roofline, và giảm byte là cách duy nhất làm chúng nhanh hơn.

Để đo, ta viết hai phiên bản của cùng chuỗi. Phiên bản trực tiếp gọi ba kernel của PyTorch, mỗi
kernel ghi vào một bộ đệm đã cấp phát sẵn, đúng mô hình ba lần đọc, ba lần ghi. Phiên bản hợp nhất
đưa **cùng hàm** cho `torch.compile`: trình biên dịch của PyTorch lần theo đồ thị tính toán, gộp
ba phép thành một vòng lặp, sinh mã C++ cho nó rồi biên dịch. Đó đúng là tối ưu đồ thị và sinh mã,
hai bước của quy trình biên dịch mà chương mô tả. Lần gọi đầu tiên mất vài giây để biên dịch.

**Dự đoán:** trên tensor lớn, phiên bản hợp nhất nhanh hơn bao nhiêu lần? Đúng 3 lần, ít hơn, hay
nhiều hơn?

In [ ]:
import torch.nn.functional as F

C = cot
dat_hat_giong()
gamma, beta_bn = torch.rand(C) + 0.5, torch.randn(C)
trung_binh, phuong_sai = torch.randn(C), torch.rand(C) + 0.5
EPS = 1e-5
# Thống kê đã đóng băng biến chuẩn hóa theo batch thành một phép nhân và một phép cộng mỗi kênh.
s = gamma / torch.sqrt(phuong_sai + EPS)
t = beta_bn - trung_binh * s
alpha, beta = 1.5, 0.25


def chuoi(x):
    x1 = torch.relu(x)                     # X'  = ReLU(X)
    x2 = torch.addcmul(t, x1, s)           # X'' = BatchNorm(X'), thống kê đã đóng băng
    return x2 * alpha + beta               # Y   = α · X'' + β


def truc_tiep_3_kernel(x, b1, b2, y):
    torch.clamp(x, min=0.0, out=b1)        # kernel 1: đọc X, ghi X'
    torch.addcmul(t, b1, s, out=b2)        # kernel 2: đọc X', ghi X''
    torch.add(beta_t, b2, alpha=alpha, out=y)  # kernel 3: đọc X'', ghi Y


beta_t = torch.tensor(beta)
t0 = time.perf_counter()
chuoi_hop_nhat = torch.compile(chuoi, dynamic=True)
x = torch.randn(256, C)
_ = chuoi_hop_nhat(x)
do_tren_may("thời gian biên dịch lần đầu", round(time.perf_counter() - t0, 1), "s")

cac_hang = [1024, 4096, 16384]
t_truc_tiep, t_hop_nhat = [], []
for R in cac_hang:
    x = torch.randn(R, C)
    b1, b2, y = torch.empty_like(x), torch.empty_like(x), torch.empty_like(x)
    truc_tiep_3_kernel(x, b1, b2, y)
    # Cả ba cách cho cùng kết quả, kể cả cách chuẩn của PyTorch cho chuẩn hóa theo batch.
    chuan = F.batch_norm(torch.relu(x), trung_binh, phuong_sai, gamma, beta_bn, training=False, eps=EPS) * alpha + beta
    assert torch.allclose(y, chuan, atol=1e-5) and torch.allclose(chuoi_hop_nhat(x), chuan, atol=1e-5)
    t_truc_tiep.append(thoi_gian(lambda: truc_tiep_3_kernel(x, b1, b2, y)))
    t_hop_nhat.append(thoi_gian(lambda: chuoi_hop_nhat(x)))
    mb = R * C * FP32 / MB
    do_tren_may(f"{mb:5.1f} MB mỗi tensor: trực tiếp", round(t_truc_tiep[-1] * 1e3, 2), "ms")
    do_tren_may(f"{mb:5.1f} MB mỗi tensor: hợp nhất", round(t_hop_nhat[-1] * 1e3, 2), "ms")
    do_tren_may(f"{mb:5.1f} MB mỗi tensor: nhanh hơn", round(t_truc_tiep[-1] / t_hop_nhat[-1], 2), "lần")

In [ ]:
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

mb_truc = [R * C * FP32 / MB for R in cac_hang]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
ax1.plot(mb_truc, [v * 1e3 for v in t_truc_tiep], "o-", color="#2a78d6", ms=6, label="trực tiếp: 3 kernel")
ax1.plot(mb_truc, [v * 1e3 for v in t_hop_nhat], "o-", color="#eb6834", ms=6, label="hợp nhất: 1 kernel")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xticks(mb_truc, [f"{v:.1f}" for v in mb_truc])
ax1.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
ax1.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax1.yaxis.set_minor_formatter(NullFormatter())
ax1.set_xlabel("cỡ một tensor (MB)")
ax1.set_ylabel("ms mỗi lần gọi (đo trên máy này)")
ax1.set_title("Cùng chuỗi ba phép, hai cách thực thi")
ax1.legend(loc="upper left", fontsize=8, frameon=False)

ti_le = [a / b for a, b in zip(t_truc_tiep, t_hop_nhat)]
ax2.bar([f"{v:.1f} MB" for v in mb_truc], ti_le, color="#1baf7a", width=0.55)
ax2.axhline(truc_tiep / hop_nhat, color="0.4", lw=1.2, ls="--")
ax2.text(1, truc_tiep / hop_nhat * 1.03, "mô hình lý tưởng của sách: 3 lần", fontsize=8, color="0.3",
         ha="center", va="bottom")
ax2.set_ylim(0, max(max(ti_le), truc_tiep / hop_nhat) * 1.25)
ax2.set_xlabel("cỡ một tensor")
ax2.set_ylabel("trực tiếp ÷ hợp nhất (đo trên máy này)")
ax2.set_title("Hợp nhất nhanh hơn bao nhiêu lần")
plt.tight_layout()
plt.show()

Trên phần lớn máy, bản hợp nhất nhanh hơn nhưng chưa tới 3 lần, đúng như lời cảnh báo của sách. Thời gian
của mỗi kernel không chỉ là byte chia cho băng thông: còn phần tính, phần phụ trội của mỗi lời
gọi, và trên một luồng CPU, chính tốc độ nạp và ghi của lõi có thể là giới hạn trước cả bộ nhớ
chính. Dù giới hạn là gì, phiên bản hợp nhất bỏ được hai lượt ghi ra và hai lượt đọc lại của X' và
X''.

Sách minh hoạ tác động của trình biên dịch học máy bằng một kịch bản ResNet-50 trên GPU, 47 ms còn
8 ms, và nói rõ đó không phải kết quả benchmark:

In [ ]:
truoc, sau = sach(47, nguon="47 ms · 8 ms · 5.9×"), sach(8, nguon="47 ms · 8 ms · 5.9×")
theo_sach("kịch bản minh hoạ ResNet-50: nhanh hơn", truoc / sau, sach=5.9, nguon="47 ms · 8 ms · 5.9×")

## Thử thêm

1. Thêm `torch.set_num_threads(1)` ở đầu phần 2, chạy lại phần 2 và 3, rồi làm lại với
   `torch.set_num_threads(2)` (Colab có 2 luồng). Trần nào nâng lên nhiều hơn, sức tính hay băng
   thông? Điểm gãy của máy dịch về phía nào?
2. Ở phần 4, kéo dài chuỗi thêm hai phép, chẳng hạn một `torch.tanh` và một phép nhân với hằng số,
   ở cả hai phiên bản. Mô hình của sách nói lưu lượng trực tiếp tăng lên năm lần đọc và năm lần
   ghi, còn bản hợp nhất vẫn một lần đọc, một lần ghi. Tỉ số đo được có tăng theo không?
3. Ở phần 3, đổi `M = N = 4096` thành `M = N = 1024`. Trọng số giờ chỉ khoảng 4 MB và có thể nằm
   gọn trong cache giữa các lần gọi. Điểm batch 1 còn nằm trên dốc của roofline không, và vì sao
   điều đó cho thấy roofline phải vẽ cho đúng tầng bộ nhớ đang cấp dữ liệu?

## Tóm lại

* Trong cùng một tầng dày đặc, phần nhân ma trận có mật độ tăng theo kích thước, còn phần theo từng
  phần tử có mật độ cố định 1/8 FLOP/byte ở FP32.
* Roofline của một máy dựng được từ hai số đo: thông lượng của phép nhân ma trận lớn và băng thông
  của một phép theo từng phần tử trên mảng lớn.
* Tầng dày đặc đo ở chín kích thước batch đi theo đúng hình dạng sách dự đoán: một dốc rồi một
  mái, và nằm dưới roofline vì roofline chỉ là cận trên.
* Với kích thước nhỏ, phụ trội cố định của mỗi lời gọi chi phối, một lý do nữa để hợp nhất.
* Hợp nhất kernel bỏ các lượt đi về của tensor trung gian; mô hình lý tưởng cho 3 lần, số đo thường
  thấp hơn vì thời gian của kernel không chỉ là lưu lượng.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Tăng tốc phần cứng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch11-hw-acceleration/), dựng từ chương
[*Hardware Acceleration*](https://mlsysbook.ai/vol1/hw_acceleration/hw_acceleration.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.